# Prepare ETTm1

Select HUFL from `ETTm1.csv` and prepare the 15-minute, hourly, six-hourly,
and daily series used in the study. Apply the study's `HUFL * 0.25`
transformation, then sum complete intervals at each coarser resolution.

Download `ETT-small/ETTm1.csv` from the
[ETTDataset repository](https://github.com/zhouhaoyi/ETDataset).
Run the cells in order from the repository root or `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "tools" / "prepare_datasets.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.prepare_datasets import write_outputs

## Input and output

Place `ETTm1.csv` in `data/Electricity/`, or update `INPUT_PATH` below.
Set `OVERWRITE = True` to replace existing prepared CSVs.

In [ ]:
INPUT_PATH = ROOT / "data" / "Electricity" / "ETTm1.csv"
OUTPUT_DIR = ROOT / "data" / "Electricity"
OVERWRITE = False

## Select HUFL

In [ ]:
raw = pd.read_csv(INPUT_PATH, usecols=["date", "HUFL"])
raw["date"] = pd.to_datetime(raw["date"])
raw = raw.sort_values("date").reset_index(drop=True)

ett_15m = pd.DataFrame({
    "date": raw["date"],
    "value": raw["HUFL"] * 0.25,
}).set_index("date")

## Aggregate and save

Keep hourly, six-hourly, and daily intervals containing exactly 4, 24,
and 96 finest-resolution observations, respectively.

In [ ]:
def aggregate_complete(frame, rule, expected_count):
    grouped = frame["value"].resample(
        rule, origin="start_day", label="left", closed="left"
    )
    totals = grouped.sum().to_frame("value")
    return totals.loc[grouped.count() == expected_count]


outputs = {
    "ETTm1_HUFL_15m.csv": ett_15m,
    "ETTm1_HUFL_1h.csv": aggregate_complete(ett_15m, "1h", 4),
    "ETTm1_HUFL_6h.csv": aggregate_complete(ett_15m, "6h", 24),
    "ETTm1_HUFL_1d.csv": aggregate_complete(ett_15m, "1D", 96),
}

In [ ]:
write_outputs(outputs, OUTPUT_DIR, overwrite=OVERWRITE)

In [ ]:
summary = pd.DataFrame([
    {"file": name, "rows": len(frame),
     "start": frame.index[0], "end": frame.index[-1]}
    for name, frame in outputs.items()
])
summary

## Training split

The study's ETTm1 test period starts at `2018-01-31 00:00:00`.
The loader uses the final 10% of the preceding observations for validation
and fits normalization on the remaining training observations.
See [Dataset preparation](../docs/DATASETS.md) for the experiment settings.